### Remember to use a GPU-based runtime!

## Datasets and Benchmarking

This lab session serves as an introduction to _Eleuther's Language Model Evaluation Harness_ - a library which is widely used to benchmark LLMs. Let's start by downloading the library and installing it.

If you want to learn more about this project, check out their github page: https://github.com/EleutherAI/lm-evaluation-harness. It is pretty useful, easy to use and has been used to evaluate numerous architectures, e.g. https://arxiv.org/pdf/2312.00752, https://arxiv.org/pdf/2307.03170.

In [1]:
! git clone https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
! git checkout fb963f0
! pip install -e .
%cd /content

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 59606, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 59606 (delta 6), reused 0 (delta 0), pack-reused 59577 (from 2)
Receiving objects: 100% (59606/59606), 32.55 MiB | 13.70 MiB/s, done.
Resolving deltas: 100% (41207/41207), done.
/content/lm-evaluation-harness
Note: switching to 'fb963f0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at fb963f0f Multimodal prototyping (#2243

Now, restart the runtime (`Runtime > Restart session`)

LM Eval Harness is very easy to use. We can - for example - evaluate some model very easily.

In [2]:
!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen2-0.5B \
    --tasks rte \
    --limit 100 \
    --output output/ \
    --log_samples

2026-01-09 11:18:00.003111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767957480.022840    2340 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767957480.028732    2340 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767957480.043655    2340 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767957480.043679    2340 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767957480.043683    2340 computation_placer.cc:177] computation placer alr

We just evaluated a huggingface model (https://huggingface.co/Qwen/Qwen2.5-0.5B) on a task (https://aclweb.org/aclwiki/Recognizing_Textual_Entailment) from the GLUE Benchmark (https://gluebenchmark.com/tasks).
We limited ourselves to just 100 testing samples.

The results of the testing procedure (the samples, our models outputs, etc.) are stored in the `./output` folder. Look at the `./output/Qwen__Qwen2.0-0.5B/samples_*.jsonl` file.

Q1. Why are there numbers associated with the responses of our model (`..."resps": [[["-1.2109375", "False"]], [["-1.0859375", "True"]]]...` ?).

To figure that out we can look at the task's definition (https://github.com/EleutherAI/lm-evaluation-harness/blob/bcb4cbf464668450f1969a4f1bef28f8638d1648/lm_eval/tasks/glue/rte/default.yaml).

Q2. Take a look at the `output_type` - how are multiple choice tasks evaluated? You can find the answer here: https://blog.eleuther.ai/multiple-choice-normalization/.

Q3. Do we need to do generation to evaluate a model on multiple-choice tasks?




A1. Numbers are sum log probabilities of given continuation (thats why it is negative and sum of exponents is lesser than 1 - other continuation are possible, but not mentioned).



A2. Metric `acc` means Unnormalized score of continuation: The score of continuation $x_{m:n_i}$ for prompt $x_{0:m}$ is determined using

$$
\sum_{i=m}^{n_i -1} \log (P(x_j|x_{0:j}))
$$


A3. No, generation tasks and multiple choice are different - we just check probability of such a continuation. Nothing needs to be genereated.

## Task - Calculator
Now we know how to evaluate models using LM Eval Harness. Let's evaluate some LLMs on a new task - _calculator_. The calculator task is very simple one but a bit unusual for a _language_ model. We want to see how good a model is at doing arithmetics (in particular addition of numbers $\in \{0, 1, 2, ... 99\}$).

To do that, we will need to create:
1. A Huggingface-compatible dataset (split into train and test set, containing all possible pairs of integers $0 \leq x, y \leq 99$).
2. A new metric, that will allow us to measure the performance of the model (unlike in multiple choice, we will now sample the response of the model and try to parse it to compare _as a number_ to the ground truth.
3. Finally, a new Harness task that ties those two things together.

If you get stuck, you can always look at the LM Eval Harness guide to adding new tasks: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md.

## The Dataset
As you may know, Huggingface datasets can come in a variety of formats. We will use probably the most straightforward one - `.jsonl`. Each line of the train / test files should contain a json `d` containing entries `num1`, `num2`, `label`, which satisfy `num1 + num2 == label`. Also, make the values of all the entries strings - this will prevent some harness errors later.


In [3]:
import json
import random

# <<<< Fill me in
import numpy as np
np.random.seed(42)

def get_sample_from_numbers(arr):
    arr = arr.astype(int)
    num1 = arr // 100
    num2 = np.remainder(arr, 100)
    label = num1 + num2
    # num1 = num1.astype(str)
    # num2 = num2.astype(str)
    # label = label.astype(str)
    return [{"num1": str(n1), "num2": str(n2), "label": str(l)} for (n1,n2,l) in zip(num1,num2,label)]
test_nrs = np.random.choice(10000, 1000, replace=False)
train_nrs = np.random.permutation(np.setdiff1d(np.arange(10000), test_nrs))

test = get_sample_from_numbers(test_nrs)
train = get_sample_from_numbers(train_nrs)

# <<<<

with open('/content/train.jsonl', 'w') as outfile:
    for entry in train:
        json.dump(entry, outfile)
        outfile.write('\n')

with open('/content/test.jsonl', 'w') as outfile:
    for entry in test:
        json.dump(entry, outfile)
        outfile.write('\n')


# loosely test if the solution is correct
def check_correctnes():
    encountered_pairs = {}
    for filename in ['train.jsonl', 'test.jsonl']:
        with open(filename, 'r') as f:
            for line in f:
                data = json.loads(line)
                assert isinstance(data['label'], str)
                assert isinstance(data['num1'], str)
                assert isinstance(data['num2'], str)
                assert int(data['label']) == int(data['num1']) + int(data['num2'])

check_correctnes()

In [4]:
!head -n 10 /content/train.jsonl

{"num1": "6", "num2": "57", "label": "63"}
{"num1": "53", "num2": "71", "label": "124"}
{"num1": "97", "num2": "4", "label": "101"}
{"num1": "86", "num2": "15", "label": "101"}
{"num1": "57", "num2": "21", "label": "78"}
{"num1": "63", "num2": "7", "label": "70"}
{"num1": "78", "num2": "57", "label": "135"}
{"num1": "13", "num2": "24", "label": "37"}
{"num1": "67", "num2": "34", "label": "101"}
{"num1": "0", "num2": "55", "label": "55"}


## The Metric
To measure our performance, we will create two custom metric - `l1` and `l2`. Each will receive the ground truth (`references`) and our model's answer (`predictions`) and return a single number, i.e. the absolute value (or the square) of the difference between the ground truth and the model's answer.

Note, that the model's answer might not be a valid number or might consist of a number followed by some additional text. Our testing procedure will be:
- first, remove the leading spaces from our model's response
- then, use the longest prefix of the answer that consists solely of digits
- if no such prefix exists - use `99` (mean / median value over the domain) as the response

The details of the procedure are quite arbitrary - unfortunately, specifics of evaluating LLMs often are.

The code for running our custom metrics needs to be put in the right place in the LM Eval Harness repository.

In [5]:
basedir = '/content/lm-evaluation-harness/lm_eval/tasks/calculator'
! mkdir -p {basedir}

with open(f'{basedir}/metrics.py', 'w') as f:
    f.write(
r"""
from typing import List

def l2(references: List[str], predictions: List[str], **kwargs) -> float:
    # You can assume references[0] is the ground truth, predictions[0] is the model's response, and **kwargs don't matter
    # <<<< Fill me in
    import re
    pred = predictions[0].lstrip()
    pred = re.findall(r'^\d+', pred)
    # pred = re.findall('^\\d+', pred)

    if len(pred) > 0:
        pred = int(pred[0])
    else:
        pred = 99
    return (pred - int(references[0])) ** 2
    # <<<<


def l1(references: List[str], predictions: List[str], **kwargs) -> float:
    # You can assume references[0] is the ground truth, predictions[0] is the model's response, and **kwargs don't matter
    # <<<< Fill me in
    # Note: I have added r at the beginning of string, otherwise \d would be interpreted as an escape sequence
    import re
    pred = predictions[0].lstrip()
    pred = re.findall(r'^\d+', pred)
    # pred = re.findall('^\\d+', pred)

    if len(pred) > 0:
        pred = int(pred[0])
    else:
        pred = 99
    return abs(pred - int(references[0]))
    # <<<<


if __name__ == "__main__":
    assert l1(["10"], ["10"]) == 0
    assert l1(["10"], ["12"]) == 2
    assert l1(["10"], ["10 abc"]) == 0
    assert l1(["10"], ["abc"]) == 89

    assert l2(["10"], ["10"]) == 0
    assert l2(["10"], ["12"]) == 4
    assert l2(["10"], ["10 abc"]) == 0
    assert l2(["10"], ["abc"]) == 7921
"""
)

# Test the solution
! python3 {basedir}/metrics.py

# The Task
Now that our dataset and metric are ready, we need to define the task. You will need to fill in the `.yaml` defining the calculator task (fill in the fragments within \<these brackets\>. If you're in doubt, try to find the relevant section in the documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md - it contains relevant examples.

In [13]:
with open(f'{basedir}/calculator.yaml', 'w') as f:
    f.write(
'''
task: calculator
doc_to_text: "What is the result of the following operation: {{num1}} + {{num2}}?\nAnswer:"
doc_to_target: "{{label}}"
dataset_path: json
dataset_name: null
dataset_kwargs:
    data_files:
        train: train.jsonl
        test: test.jsonl
training_split: train
validation_split: null
test_split: test
metric_list:
  - metric: !function metrics.l1
    higher_is_better: false
    aggregation: mean
  - metric: !function metrics.l2
    higher_is_better: false
    aggregation: mean
''')

Now that our task is ready we can run it.

In [14]:
!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen2-0.5B \
    --tasks calculator \
    --output output/ \
    --limit 10 \
    --log_samples

2026-01-09 11:36:20.300272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767958580.320547    7272 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767958580.326593    7272 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767958580.342678    7272 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767958580.342716    7272 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767958580.342720    7272 computation_placer.cc:177] computation placer alr

Why are we getting so bad scores? Look in the `./output` folder. What is wrong with the outputs? Can we fix this somehow?

Queries are strings "... <num1> + <num2>", where <num{i}> is literally just string, not an true number from "doc[num{i}]".

To solve this I have changed doc_to_text to use !function. However the result was bad - there are a lot of comments before equation, e.g., 62+52=114 in one generated sample. So I have also asked model for no comments. Here are results (should be good, but Qwen is not listening to orders, it's responses are e.g. "62 + 52 = 114"):

In [8]:
# with open(f'{basedir}/calculator.py', 'w') as f:
#     f.write(
# '''
# def doc_to_text(doc):
#     return (
#         f"What is the result of the following operation: "
#         f"{doc['num1']} + {doc['num2']}?"
#         # f"Answer (NO COMENTS, no equation, just a correct number (e.g., for equation 12+5 answer with single short answer: '17', and NOT '12 + 5 = 17')):"
#     )
# ''')

In [15]:
with open(f'{basedir}/calculator.yaml', 'w') as f:
    f.write(
'''
task: calculator
doc_to_text: "What is the result of the following operation: {{num1}} + {{num2}}?\nAnswer: {{num1}} + {{num2}}="
doc_to_target: "{{label}}"
dataset_path: json
dataset_name: null
dataset_kwargs:
    data_files:
        train: train.jsonl
        test: test.jsonl
training_split: train
validation_split: null
test_split: test
metric_list:
  - metric: !function metrics.l1
    higher_is_better: false
    aggregation: mean
  - metric: !function metrics.l2
    higher_is_better: false
    aggregation: mean
''')

In [16]:
!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen2-0.5B \
    --tasks calculator \
    --output output/ \
    --limit 10 \
    --log_samples

2026-01-09 11:38:27.602526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767958707.633919    7873 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767958707.643692    7873 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767958707.667378    7873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767958707.667429    7873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767958707.667439    7873 computation_placer.cc:177] computation placer alr